# EcoRewind — build and deploy the website (Colab **T4 GPU**, High-RAM; ~1 h, ~2 CCU)

Run **after** `51_phase5_analysis_colab.ipynb` (reports + demo export on Drive).
1. trains + exports the live-lab SimVP (ONNX), 2. converts the demo export into web data, 3. builds the site,
4. publishes it to GitHub Pages (branch `gh-pages`). Every number and map on the site comes from your runs.

### 1. Setup

In [1]:
from google.colab import drive, userdata
drive.mount('/content/drive')
import os, glob, shutil, time
TOKEN = userdata.get('GITHUB_TOKEN')
REPO = '/content/ecorewind2'
if os.path.isdir(REPO):
    !git -C {REPO} pull -q
else:
    !git clone -q https://{TOKEN}@github.com/DAISHINKAN7/EcoRewind2.git {REPO}
%cd {REPO}
!git log --oneline -1
!pip -q install -r requirements.txt
D = '/content/drive/MyDrive/EcoRewind2'
X = '/content/drive/MyDrive/EcoRewind2_exports'
!python scripts/cubes_cache.py --drive {D}/cubes --local /content/cubes --exports-drive {X} 2>&1 | tail -4
!python scripts/copy_ppost.py {X} /content/ppost


Mounted at /content/drive
/content/ecorewind2
b92e3ac (HEAD -> main, origin/main, origin/HEAD) web: deployable anywhere; cleanup
everglades_A: T=33 shape=(2202, 2199) hash faa5c71d42f4
everglades_B: T=42 shape=(734, 733) hash c8caab5a6167
mississippi_A: T=33 shape=(2238, 2235) hash 437a79d69c5f
mississippi_B: T=42 shape=(746, 745) hash 506e2793ee02
retry everglades_B_S1_2016Q3_PPOST.tif OSError
PPOST files ok: 12


### 2. Live-lab model (T4, ~25 min, once)

In [2]:
!git -C /content/ecorewind2 pull -q && git -C /content/ecorewind2 log --oneline -1
%cd /content/ecorewind2
!python -u scripts/export_lab_model.py --cubes /content/cubes --phase1 {D}/phase1 --table-cache /content/tables \
    --ppost-exports /content/ppost --out {D}/web_data/lab > /content/lab.log 2>&1; echo exit=$?
!tail -40 /content/lab.log
!ls -lh {D}/web_data/lab

b92e3ac (HEAD -> main, origin/main, origin/HEAD) web: deployable anywhere; cleanup
/content/ecorewind2
exit=0
      ep35 step 120/150  loss 0.0443  6.8 it/s  epoch ETA 4s
      ep35 step 150/150  loss 0.0441  6.9 it/s  epoch ETA 0s
    ecotransformer_s0 ida k0 ep35/40: loss 0.0441  val NDVI R² 0.588 (best 0.588)  772s elapsed, <= 1.8 min left  [GPU mem 1.2 GB]
      ep36 step 30/150  loss 0.0395  6.2 it/s  epoch ETA 19s
      ep36 step 60/150  loss 0.0432  6.5 it/s  epoch ETA 14s
      ep36 step 90/150  loss 0.0414  6.6 it/s  epoch ETA 9s
      ep36 step 120/150  loss 0.0458  6.6 it/s  epoch ETA 5s
      ep36 step 150/150  loss 0.0374  6.6 it/s  epoch ETA 0s
    ecotransformer_s0 ida k0 ep36/40: loss 0.0374  val NDVI R² 0.582 (best 0.588)  795s elapsed, <= 1.5 min left  [GPU mem 1.2 GB]
      ep37 step 30/150  loss 0.0378  7.3 it/s  epoch ETA 16s
      ep37 step 60/150  loss 0.0404  7.1 it/s  epoch ETA 13s
      ep37 step 90/150  loss 0.0372  6.9 it/s  epoch ETA 9s
      ep37 step 120/

In [7]:
!python -u scripts/export_lab_model.py --cubes /content/cubes --phase1 {D}/phase1 --table-cache /content/tables \
    --ppost-exports /content/ppost --out {D}/web_data/lab --tiles-only > /content/tiles.log 2>&1; echo exit=$?
!tail -5 /content/tiles.log
!ls {D}/web_data/lab/tiles | wc -l

exit=2
                           [--phase3-config PHASE3_CONFIG]
                           [--phase1-config PHASE1_CONFIG] [--fast]
                           [--archs ARCHS [ARCHS ...]] [--tiles]
                           [--tile-min-cover TILE_MIN_COVER]
export_lab_model.py: error: unrecognized arguments: --tiles-only
ls: cannot access '/content/drive/MyDrive/EcoRewind2/web_data/lab/tiles': No such file or directory
0


In [8]:
!git -C /content/ecorewind2 pull -q && git -C /content/ecorewind2 log --oneline -1
%cd /content/ecorewind2
!python -u scripts/export_lab_model.py --cubes /content/cubes --phase1 {D}/phase1 --table-cache /content/tables \
    --ppost-exports /content/ppost --out {D}/web_data/lab --tiles-only > /content/tiles.log 2>&1; echo exit=$?
!tail -5 /content/tiles.log
!ls {D}/web_data/lab/tiles | wc -l

3fda2f5 (HEAD -> main, origin/main, origin/HEAD) lab export: --tiles-only adds tiles to an existing export without retraining
/content/ecorewind2
exit=0
17 tiles (cover >= 0.3)
wrote 17 tiles into /content/drive/MyDrive/EcoRewind2/web_data/lab
34


### 3. Web data (map overlays + results)

In [14]:
!python scripts/build_web_data.py --reports {D}/reports --demo {D}/demo --lab {D}/web_data/lab --out web/public/data
!du -sh web/public/data; ls web/public/data

{'generated': '…', 'registry': 22, 'leaderboard': 27, 'verdict': 52, 'space': 64, 'attribution': 60, 'external': 180, 'phase5': 30, 'carbon': '…', 'compute': 8}
/content/ecorewind2/scripts/build_web_data.py:308: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = Image.fromarray(colourise(a[0], kind), "RGBA")
everglades_A: 146 layers
everglades_B: 851 layers
mississippi_A: 354 layers
mississippi_B: 679 layers
wrote web/public/data
912M	web/public/data
lab  layers  maps.json	pixels	results.json  training


In [10]:
import json, os, glob
D = '/content/drive/MyDrive/EcoRewind2'
LAB = f'{D}/web_data/lab'
WEB = '/content/ecorewind2/web/public/data'          # written by cell 3 (only exists in this runtime)
ok = lambda c, msg: print(('PASS ' if c else 'FAIL ') + msg)
size = lambda p: sum(os.path.getsize(f) for f in glob.glob(p + '/**', recursive=True) if os.path.isfile(f)) / 1e6

print('— Lab export (Drive) —')
m = json.load(open(f'{LAB}/meta.json')) if os.path.exists(f'{LAB}/meta.json') else {}
models = [x['id'] for x in m.get('models', [])]
ok(set(models) >= {'simvp', 'convlstm', 'ecotransformer'}, f'networks in meta: {models or "none (old export)"}')
for a in ['simvp', 'convlstm', 'ecotransformer']:
    f = f'{LAB}/{a}.onnx'
    ok(os.path.exists(f), f'{a}.onnx ' + (f'{os.path.getsize(f)/1e6:.1f} MB' if os.path.exists(f) else 'missing'))
ok(len(m.get('tiles', [])) > 0, f"tiles: {len(m.get('tiles', []))} in meta, {len(glob.glob(LAB + '/tiles/*_x.f16'))} files")
ok(any(s for s in m.get('static_stats', []) if s), 'static normalisation constants present')
ok(all(os.path.exists(f'{LAB}/patches/p0_ref_{a}.f16') for a in models) if models else False, 'per-network reference outputs for the patches')
print(f"   lab size {size(LAB):.0f} MB, track {m.get('track')}, val R² " + ', '.join(f"{x['id']} {x['val_r2']:.3f}" for x in m.get('models', []) if x.get('val_r2') is not None))

print('\n— Built web data (this runtime) —')
if not os.path.exists(f'{WEB}/results.json'):
    print('FAIL web/public/data not found: rerun cell 3 (build_web_data) in this runtime')
else:
    r = json.load(open(f'{WEB}/results.json')); mp = json.load(open(f'{WEB}/maps.json'))
    ok(len(r.get('compute', [])) > 0, f"compute log rows: {len(r.get('compute', []))}")
    ok(len(r.get('verdict', [])) >= 52, f"placebo verdicts: {len(r.get('verdict', []))}")
    ok(len(r.get('phase5', [])) >= 30, f"phase 5 rows: {len(r.get('phase5', []))}")
    for st, v in mp.items():
        print(f"   {st}: {len(v['models'])} models, {len(v['layers'])} layers")
    ok(sum(len(v['models']) for v in mp.values()) >= 46, 'model maps for all sites (6 + 19 + 10 + 11)')
    px = glob.glob(f'{WEB}/pixels/*.json')
    big = max((os.path.getsize(f) for f in px), default=0) / 1e6
    ok(len(px) == 4 and big < 20, f'pixel samples: {len(px)} files, largest {big:.0f} MB (slimmed if < 20 MB)')
    wl = json.load(open(f'{WEB}/lab/meta.json')) if os.path.exists(f'{WEB}/lab/meta.json') else {}
    ok(len(wl.get('models', [])) == 3, f"lab copied into the site: {len(wl.get('models', []))} networks, {len(glob.glob(WEB + '/lab/tiles/*'))} tile files")
    ok(os.path.exists('/content/ecorewind2/web/dist/index.html'), 'site built (web/dist)')
    print(f'   total web data {size(WEB):.0f} MB')

print('\n— Published (gh-pages) —')
!git -C /content/ecorewind2 ls-remote origin gh-pages

— Lab export (Drive) —
PASS networks in meta: ['simvp', 'convlstm', 'ecotransformer']
PASS simvp.onnx 3.5 MB
PASS convlstm.onnx 2.8 MB
PASS ecotransformer.onnx 9.3 MB
PASS tiles: 17 in meta, 17 files
PASS static normalisation constants present
PASS per-network reference outputs for the patches
   lab size 140 MB, track B, val R² simvp 0.650, convlstm 0.636, ecotransformer 0.588

— Built web data (this runtime) —
PASS compute log rows: 8
PASS placebo verdicts: 52
PASS phase 5 rows: 30
   everglades_A: 6 models, 146 layers
   everglades_B: 19 models, 851 layers
   mississippi_A: 10 models, 354 layers
   mississippi_B: 11 models, 679 layers
PASS model maps for all sites (6 + 19 + 10 + 11)
PASS pixel samples: 4 files, largest 15 MB (slimmed if < 20 MB)
PASS lab copied into the site: 3 networks, 34 tile files
PASS site built (web/dist)
   total web data 951 MB

— Published (gh-pages) —
afc9ad627ce5086ea6bb7809cf21db8bbe757109	refs/heads/gh-pages


### 4. Build the site

In [11]:
!node -v 2>/dev/null || (curl -fsSL https://deb.nodesource.com/setup_20.x | bash - >/dev/null && apt-get install -y nodejs >/dev/null)
!node -v && cd web && npm ci --no-audit --no-fund --silent && npm run build --silent && du -sh dist

v22.23.4
v22.23.4
vite v5.4.8 building for production...
✓ 128 modules transformed.
node_modules/onnxruntime-web/dist/ort.bundle.min.mjs (6:95285): Use of eval in "node_modules/onnxruntime-web/dist/ort.bundle.min.mjs" is strongly discouraged as it poses security risks and may cause issues with minification.
dist/index.html                                        0.94 kB │ gzip:   0.52 kB
dist/assets/ort-wasm-simd-threaded-B92PF46Y.wasm  11,018.73 kB
dist/assets/index-B1WZTgb_.css                        90.83 kB │ gzip:  18.72 kB
dist/assets/raw-BwxNWhoh.js                            0.14 kB │ gzip:   0.14 kB
dist/assets/deflate-BVCD62Q3.js                        0.20 kB │ gzip:   0.18 kB
dist/assets/packbits-BR4MpSA5.js                       0.37 kB │ gzip:   0.28 kB
dist/assets/webimage-COfr_kk3.js                       0.71 kB │ gzip:   0.41 kB
dist/assets/lzw-JjwZRZ-9.js                            1.44 kB │ gzip:   0.82 kB
dist/assets/jpeg-PBo4UUAo.js                           9.01 k

### 5. Publish to GitHub Pages

In [12]:
!cd web/dist && touch .nojekyll && rm -rf .git && git init -q && git checkout -q -b gh-pages && \
  git -c user.name="EcoRewind build" -c user.email="build@users.noreply.github.com" add -A >/dev/null && \
  git -c user.name="EcoRewind build" -c user.email="build@users.noreply.github.com" commit -q -m "site build" && \
  git push -q -f https://{TOKEN}@github.com/DAISHINKAN7/EcoRewind2.git gh-pages && echo PUBLISHED
print('Site: https://daishinkan7.github.io/EcoRewind2/  (first time: repo Settings → Pages → Source: branch gh-pages / root)')

PUBLISHED
Site: https://daishinkan7.github.io/EcoRewind2/  (first time: repo Settings → Pages → Source: branch gh-pages / root)


**Runtime → Disconnect and delete runtime** when done.

In [13]:
!python -u scripts/export_lab_model.py --cubes /content/cubes --phase1 {D}/phase1 --table-cache /content/tables \
    --ppost-exports /content/ppost --out {D}/web_data/lab --tiles-only > /content/tiles.log 2>&1; echo exit=$?
!tail -3 /content/tiles.log


exit=0
17 tiles (cover >= 0.3)
wrote 17 tiles into /content/drive/MyDrive/EcoRewind2/web_data/lab


In [ ]:
import os
from google.colab import userdata
os.environ["CLOUDFLARE_API_TOKEN"] = userdata.get("CLOUDFLARE_API_TOKEN")
os.environ["CLOUDFLARE_ACCOUNT_ID"] = userdata.get("CLOUDFLARE_ACCOUNT_ID")
DATA = "/content/ecorewind2/web/public/data"
open(f"{DATA}/_headers", "w").write("/*\n  Access-Control-Allow-Origin: *\n  Cache-Control: public, max-age=3600\n/lab/*\n  Cache-Control: public, max-age=86400\n")
!node -v 2>/dev/null || (curl -fsSL https://deb.nodesource.com/setup_20.x | bash - >/dev/null && apt-get install -y nodejs >/dev/null)
!npx --yes wrangler@4 pages project create ecorewind-data --production-branch main || true   # first time only
!npx --yes wrangler@4 pages deploy {DATA} --project-name ecorewind-data --branch main --commit-dirty=true